In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql('use catalog ecommerce_data')

spark.sql('create schema if not exists gold')

gold_run_id=str(uuid.uuid4())

run_ts_str = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')

run_date_str=datetime.utcnow().strftime('%Y-%m-%d')

print('Current Gold run id',gold_run_id)
print('Current Timestamp folder',run_ts_str)

In [0]:
%sql
create table if not exists ecommerce_data.gold.gold_control(
    layer string,
    entity_name string,
    last_processed_silver_run_id string,
    last_processed_silver_timestamp timestamp,
    rows_merged bigint,
    run_status string,
    gold_run_id string,
    updated_at timestamp
)
using Delta

3 helper functions 
1. To update the new incoming data from silver
2. To fetch the latest watermark
3. To update the control table

1 -> get latest watermark()

In [0]:
def get_last_processed_ingested_at(entity_name):
    gold_df=spark.table('ecommerce_data.gold.gold_control')
    ctrl=(gold_df.filter(
        (f.col('layer') == 'Gold') &
        (f.col('entity_name') == entity_name) &
        (f.col('run_status')=='success')
    )
    .orderBy(f.col('updated_at').desc())
    .limit(1))

    rows=ctrl.collect()

    if not rows:
        return None
    return rows[0]['last_processed_silver_timestamp']


2 -> upsert_to_silver()

In [0]:
def upsert_to_gold(source_df,target_table,join_id):
    if spark.catalog.tableExists(target_table):
        df=DeltaTable.forName(spark,target_table)
        (
        df.alias('t').merge(
            source_df.alias('s'),f"t.{join_id}=s.{join_id}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    else:
        source_df.write.format('delta').mode('append').saveAsTable(target_table)     


3 -> update control table

In [0]:
def upsert_control(entity_name,last_processed_silver_run_id,last_processed_silver_timestamp,rows_merged):
    control_df=spark.createDataFrame([
        (
            'gold',
            entity_name,
            str(last_processed_silver_run_id),
            last_processed_silver_timestamp,
            int(rows_merged),
            'success',
            str(gold_run_id),
            datetime.utcnow()
        )],
        schema="""
            layer string,
            entity_name string,
            last_processed_silver_run_id string,
            last_processed_silver_timestamp timestamp,
            rows_merged int,
            run_status string,
            gold_run_id string,
            updated_at timestamp
        """
    )

    target_table=DeltaTable.forName(spark,'ecommerce_data.gold.gold_control')
    (
    target_table.alias('t').merge(
        control_df.alias('s'),
        "t.layer=s.layer and t.entity_name=s.entity_name"
    ).
    whenMatchedUpdate(set={
        "t.last_processed_silver_run_id":"s.last_processed_silver_run_id",
        't.last_processed_silver_timestamp':'s.last_processed_silver_timestamp',
        "t.rows_merged":"s.rows_merged",
        "t.run_status":"s.run_status",
        "t.gold_run_id":"s.gold_run_id",
        "t.updated_at":"s.updated_at"
    }).
    whenNotMatchedInsertAll().
    execute())

Get only the changed data

In [0]:
last_gold_ts=get_last_processed_ingested_at('orders information')

print('last processed timestamp for gold:',last_gold_ts)

current_changed_orders=spark.read.table('ecommerce_data.silver.orders_transformed')
current_changed_products=spark.read.table('ecommerce_data.silver.products_transformed')
current_changed_payments=spark.read.table('ecommerce_data.silver.payments_transformed')

if last_gold_ts is None:
    changed_orders=current_changed_orders
    changed_products=current_changed_products
    changed_payments=current_changed_payments

else:
    changed_orders=current_changed_orders.filter(f.col('updated_at')>f.lit(last_gold_ts))
    changed_products=current_changed_products.filter(f.col('updated_at')>f.lit(last_gold_ts))
    changed_payments=current_changed_payments.filter(f.col('updated_at')>f.lit(last_gold_ts))   


changed_orders_count=changed_orders.count()
changed_products_count=changed_products.count()
changed_payments_count=changed_payments.count()

print('changed orders count:',changed_orders_count)
print('changed products count:',changed_products_count)
print('changed payments count:',changed_payments_count)


Find the impacted order id's

In [0]:
impacted_from_orders=changed_orders.select('order_id').distinct()
impacted_from_payments=changed_payments.select('order_id').distinct()
impacted_from_products=(
    changed_products.alias('t').join(current_changed_orders.alias('s'),
    f.col('t.product_id')==f.col('s.product_id'),
    'inner')
    .select('s.order_id').distinct()
)

impacted_order_ids=(
    impacted_from_orders.union(impacted_from_payments).union(impacted_from_products).distinct()
)

print('impacted order ids count:',impacted_order_ids.count())



Gold delta  for impacted order id's

In [0]:
impacted_orders=(
    current_changed_orders.alias('o')
    .join(impacted_order_ids,'order_id','inner'))

gold_delta=(
    impacted_orders.alias('o')
    .join(current_changed_products.alias('p'),f.col('o.product_id')==f.col('p.product_id'),'inner')
    .join(current_changed_payments.alias('py'),f.col('o.order_id')==f.col('py.order_id'),'inner')
    .select(
    f.col('o.order_id'),
    f.col('o.customer_id'),
    f.col('p.product_id'),
    f.col('p.product_name'),
    f.col('p.category'),
    f.col('p.price').alias('product_price'),
    f.col('o.order_status'),
    f.col('o.order_amount'),
    f.col('py.payment_id'),
    f.col('py.payment_status'),
    f.col('py.paid_amount'),
    f.col('o.order_date'),
    f.col('o.year'),
    f.col('o.month'),
    f.greatest(
        f.col('o.updated_at').cast('timestamp'),
        f.col('p.updated_at').cast('timestamp'),
        f.col('py.processed_at').cast('timestamp')
    ).alias('gold_update_ts'))
    .withColumn(
        "payment_completion_ratio",
        f.when(f.col('order_amount')>0,f.col('paid_amount')/f.col('order_amount'))
        .otherwise(f.lit(0.0))
    )
    .withColumn(
        'payment_state',
        f.when(f.col('order_amount')==0,'Invalid Order Amount')
        .when(f.col('payment_completion_ratio')==0,'Unpaid')
        .when(f.col('payment_completion_ratio')==1,'Paid')
        .when(f.col('payment_completion_ratio')<1,'Partially Paid')
        .when(f.col('payment_completion_ratio')>1,'Overpaid')
        .otherwise('Unknown')
    )
    .withColumn('gold_ingested_at',f.current_timestamp())
    .withColumn('gold_updated_date',f.to_date(f.col('gold_update_ts')))
    .withColumn('gold_run_id',f.lit(gold_run_id))
)

print('gold delta rows:',gold_delta.count())

display(gold_delta)


Merge gold current-state table

In [0]:
if gold_delta.count()>0:
    upsert_to_gold(gold_delta,'ecommerce_data.gold.orders_information','order_id')
else:
    print('no new rows to insert into gold')    

SCD 2

In [0]:
if not spark.catalog.tableExists('ecommerce_data.gold.orders_information_scd2'):
    spark.sql("""
              create table ecommerce_data.gold.orders_information_scd2
              using delta
              as
              select *,
              cast(null as timestamp) as valid_to_ts,
              cast(null as timestamp) as valid_from_ts,
              true as is_current
              from ecommerce_data.gold.orders_information
              where 1 = 0
              """)

if gold_delta.count()>0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql("""
              merge into ecommerce_data.gold.orders_information_scd2 as t 
              using gold_delta_view as s 
              on t.order_id = s.order_id and t.is_current=true
              when matched and(
                  not(t.order_status <=> s.order_status) or
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.product_price <=> s.product_price))
                  then update set
                  t.is_current = false,
                  valid_to_ts = s.gold_update_ts         
              """)
    

    spark.sql("""
              insert into ecommerce_data.gold.orders_information_scd2
              select s.*,
              s.gold_update_ts as valid_from_ts,
              cast(null as timestamp) as valid_to_ts,
              true as is_current
              from gold_delta_view s
              left join ecommerce_data.gold.orders_information_scd2 t
              on s.order_id = t.order_id and t.is_current = true
              where t.order_id is null or(
                  not(t.order_status <=> s.order_status) or
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.product_price <=> s.product_price)
              )
""")

create category table

In [0]:
if gold_delta.count()>0:
    impacted_categories=gold_delta.select('category').filter(f.col('category').isNotNull()).distinct()

    category_preference_delta=(
        spark.read.table('ecommerce_data.gold.orders_information')
        .join(impacted_categories,'category','inner')
        .groupBy('category')
        .agg(f.countDistinct('order_id').alias('total_orders'),
             f.sum(
                 f.when(f.col('order_amount')>0,f.col('order_amount'))
                 .otherwise(f.lit(0.0))
             ).alias('Gross_Merchandise_Value'),
             f.sum(
                 f.when(f.col('paid_amount')>0,f.col('paid_amount'))
                 .otherwise(f.lit(0.0))
              ).alias('Total_Paid_Amount'),
             f.avg(f.col('payment_completion_ratio')).alias('Average_Payment_Completion_Ratio'),
             (
                 f.sum(f.when(f.col('payment_status')=='failed',1).otherwise(0)) / f.count('*')
             ).alias('Payment_Failure_Rate')
             ))
    

    upsert_to_gold(category_preference_delta,'ecommerce_data.gold.category_performance','category')

Publish gold snapshots to volume

In [0]:
%sql
create volume if not exists ecommerce_data.gold.gold_snapshots_vol

In [0]:
latest_orders_path=(
    '/Volumes/ecommerce_data/gold/gold_snapshots_vol/gold_latest/orders_information/'
)

latest_category_path=(
    '/Volumes/ecommerce_data/gold/gold_snapshots_vol/gold_latest/category_performance/'
)

historical_orders_path=(
    '/Volumes/ecommerce_data/gold/gold_snapshots_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}'
)

historical_category_path=(
    '/Volumes/ecommerce_data/gold/gold_snapshots_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}'
)

spark.read.table('ecommerce_data.gold.orders_information').write.format('parquet').mode('overwrite').save(latest_orders_path)
spark.read.table('ecommerce_data.gold.category_performance').write.format('parquet').mode('overwrite').save(latest_category_path)

spark.read.table('ecommerce_data.gold.orders_information').write.format('parquet').mode('overwrite').save(historical_orders_path)
spark.read.table('ecommerce_data.gold.category_performance').write.format('parquet').mode('overwrite').save(historical_category_path)

print("latest orders path:",latest_orders_path)
print("latest category path:",latest_category_path)
print("historical orders path:",historical_orders_path)
print("historical category path:",historical_category_path)

Update control table

In [0]:
last_updated_ts=current_changed_orders.agg(f.max("updated_at").alias('mx')).collect()[0]['mx']

last_updated_id=(current_changed_orders.filter(f.col('updated_at')==last_updated_ts).agg(f.max('silver_run_id').alias('mx')).collect()[0]['mx'])if last_updated_ts is not None else None


upsert_control('orders_information',last_updated_id,last_updated_ts,gold_delta.count())

display(spark.read.table('ecommerce_data.gold.gold_control'))